In [1]:
import pandas as pd
import os
import xmltodict
import ttkbootstrap as tb
from ttkbootstrap.constants import *
from ttkbootstrap.dialogs import Messagebox, Querybox
from tkinter import filedialog
from pathlib import Path

def pegar(dados, caminho, padrao=''):
    for chave in caminho:
        if isinstance(dados, dict) and chave in dados:
            dados = dados[chave]
        else:
            return padrao
    return dados

def regime(dados, caminho):
    for chave in caminho:
        if isinstance(dados, dict) and chave in dados:
            dados = dados[chave]
            if dados == '1':
                dados = 'Simples'
            elif dados == '2':
                dados = 'SN, Sublimite RB'
            elif dados == '3':
                dados = 'Normal'
            else:
                dados = 'MEI'
        return dados

notas = []
ctes = []
erros = []

def leitor_xml(pasta_xml):
    
    """
    Função para percorrer todos os XMLs em uma pasta para processalos e gerar um DataFra,e

    Parametros:
        pasta_xml (str): Caminho para os arquivos.

    Retorno:
        DF dos clientes, Fornecedores e Produtos contidos nos XMLs.
    """

    caminho = Path(pasta_xml)
    xmls = list(caminho.rglob("*.xml"))
    notas = []
    ctes = []
    erros = []

    for arquivo in xmls:
        

        try:

            with open(arquivo, "r", encoding="utf-8") as file:
                dados = xmltodict.parse(file.read())
                if 'cteProc' in dados:
                    pass #tratativa de CTEs
                elif 'nfeProc' in dados:
                    infNFe = dados['nfeProc']['NFe']['infNFe']
                    ide = infNFe['ide']
                    emit = infNFe['emit']
                    dest = infNFe['dest']
                    det = pegar(infNFe, ['det'], [])

                    if isinstance(det, dict):
                        det = [det]

                    for item in det: 

                        prod = pegar(item,['prod'])
                        icms = pegar(item, ['imposto','ICMS'])
                        difal = pegar(item,['imposto','ICMSUFDest'])
                        ipi = pegar(item, ['imposto','IPI'])
                        pis = pegar(item,['imposto','PIS'])
                        cofins = pegar(item,['imposto','COFINS'])
                        ibscbs = pegar(item, ['imposto','IBSCBS'])
                        cst_icm = next(iter(icms.keys()))



                        notas.append({
                             #ide:
                            'Modelo': pegar(ide, ['mod']),
                            'Numero NF': pegar(ide, ['nNF']),
                            'Serie NF': pegar(ide, ['serie']),
                            'Emissão': pegar(ide, ['dhEmi']),
                            'Finalidade': pegar(ide, ['finNFe']),
                             # Emit:
                            'CNPJ Emitente': pegar(emit, ['CNPJ']),
                            'Razão Social Emitente': pegar(emit, ['xNome']),
                            'IE Emitente': pegar(emit, ['IE'], padrao="ISENTO"),
                            'UF Emitente': pegar(emit, ['enderEmit', 'UF']),
                            'Mun Emitente': pegar(emit,['enderEmit', 'xMun']),
                            'Regime Trib': regime(emit, ['CRT']),

                             #Dest: 
                            'CNPJ Destinatário': pegar(dest, ['CNPJ']),
                            'Razão Social Destinatário': pegar(dest, ['xNome']),
                            'IE Destinatário': pegar(dest, ['IE'], padrao="ISENTO"),
                            'UF Destinatário': pegar(dest, ['enderDest', 'UF']),
                            'Mun Destinatário': pegar(dest,['enderDest', 'xMun']),

                            #prod:
                            'item XML' : item['@nItem'],
                            'Cod.Produto': pegar(item,['prod','cProd']),
                            'Descrição': pegar(item,['prod','xProd']),
                            'NCM': pegar(item,['prod','NCM']),
                            'UM': pegar(item,['prod','uCom']),
                            'CFOP': pegar(item,['prod','CFOP']),
                            'Quantidade': pegar(item,['prod','qCom'], padrao=0),
                            'Valor unitário': pegar(item,['prod','vUnCom'], padrao=0),
                            'Total': pegar(item,['prod','vProd'],0),
                            'cBenef': pegar(item,['prod','cBenef'],''),

                            #tributos:
                            #ICMS:
                            'Origem' :  pegar(icms, [cst_icm, 'orig'],0),
                            'CST ICMS': pegar(icms, [cst_icm, 'CST']) if 'CST' in icms[cst_icm] else pegar(icms, [cst_icm,'CSOSN']),
                            'BC ICMS': pegar(icms, [cst_icm, 'vBC'],0),
                            '% ICMS' : pegar(icms, [cst_icm, 'pICMS'],0),
                            'Valor ICMS' : pegar(icms, [cst_icm, 'vICMS'],0),

                            #ICMS Especifico:
                            #Diferimento:
                            '% Diferimento ICMS': pegar(icms, [cst_icm, 'pDif'],0),
                            '% Redução BC': pegar(icms, [cst_icm, 'pRedBC'],0),

                            #ICMS ST: 

                            'MVA ICMS ST': pegar(icms, [cst_icm, 'pMVAST'],0), 
                            'Base ICMS ST': pegar(icms, [cst_icm, 'vBCST'],0),
                            '% ICMS ST': pegar(icms, [cst_icm, 'pICMSST'],0) if 'pICMSST' in icms[cst_icm] else pegar(icms, [cst_icm, 'vBCSTRet'],0), 
                            'Valor ICMS ST': pegar(icms, [cst_icm, 'vICMSST'],0), 
                            'Valor ICMS Retido': pegar(icms, [cst_icm,'vICMSSTRet'],0),

                            #ICMS Monofásico:
                            'Base ICMS Monofasico' : pegar(icms, [cst_icm,'qBCMono'],0),
                            'AdRem ICMS Monofasico' : pegar(icms, [cst_icm,'adRemICMS'],0),
                            'Valor ICMS Monofasico' : pegar(icms, [cst_icm,'adRemICMS'],0),

                            #IPI:
                            'Enq.IPI': pegar(ipi, ['cEnq'], padrao='999'),

                            'CST IPI': pegar(ipi, ['IPITrib', 'CST'],0) if 'IPITrib' in ipi else pegar(ipi, ['IPINT', 'CST'],0),
                            'Base IPI': pegar(ipi, ['IPITrib', 'vBC'],0),
                            '% IPI' : pegar(ipi, ['IPITrib','pIPI'],0),
                            'Valor do IPI' : pegar(ipi, ['IPITrib','vIPI'],0),

                            #PIS

                            'CST PIS': pegar(pis, [next(iter(pis.keys())), 'CST'],0),
                            'Base PIS': pegar(pis, [next(iter(pis.keys())), 'vBC'], 0),
                            '% PIS': pegar(pis, [next(iter(pis.keys())), 'pPIS'], 0),
                            'Valor PIS': pegar(pis, [next(iter(pis.keys())), 'vPIS'], 0),

                            #COFINS
                            'CST COFINS': pegar(cofins, [next(iter(pis.keys())), 'CST']),
                            'Base COFINS': pegar(cofins, [next(iter(pis.keys())), 'vBC'], 0),
                            '% COFINS': pegar(cofins, [next(iter(pis.keys())), 'pCOFINS'], 0),
                            'Valor COFINS': pegar(cofins, [next(iter(pis.keys())), 'vCOFINS'], 0),

                            #IBS e CBS:

                            'CST IBS e CBS': pegar(ibscbs, ['CST'], 'SEM DESTAQUE'),
                            'CCLASTRIB': pegar(ibscbs, ['cClassTrib']),
                            'Base IBS e CBS': pegar(ibscbs, ['gIBSCBS', 'vBC'],0),
                            'Aliq. IBSUF': pegar(ibscbs, ['gIBSCBS', 'gIBSUF', 'pIBSUF'],0),
                            'valor IBSUF': pegar(ibscbs, ['gIBSCBS', 'gIBSUF', 'vIBSUF'],0),
                            'Aliq. IBSMun': pegar(ibscbs, ['gIBSCBS', 'gIBSMun', 'pIBSMun'],0),
                            'valor IBSMun': pegar(ibscbs, ['gIBSCBS', 'gIBSMun', 'vIBSMun'],0),
                            'Aliq. CBS': pegar(ibscbs, ['gIBSCBS', 'gCBS', 'pCBS'],0),
                            'Valor CBS': pegar(ibscbs, ['gIBSCBS', 'gCBS', 'vCBS'],0),

                        })
        except:
            display(arquivo)

    processados = pd.DataFrame(notas)
    processados['Emissão'] = pd.to_datetime(processados['Emissão'], utc=True).dt.strftime('%d/%m/%Y')
    return processados

In [28]:
from pathlib import Path
caminho = Path(r"E:\OLIM AGRO - JACINTO MACHADO 03.451.1170005-03")

xmls = list(caminho.rglob("*.xml"))

print(len(xmls))

30953


In [ ]:
pasta_xml = Path(r"E:\OLIM AGRO - JACINTO MACHADO 03.451.1170005-03")

processados = leitor_xml(pasta_xml)

display(processados['CNPJ Destinatário'])

WindowsPath('E:/OLIM AGRO - JACINTO MACHADO 03.451.1170005-03/01-11-2025 30-11-2025 RECEBIDOS/42250948402634000105550010000069611150856547-nfe.xml')

WindowsPath('E:/OLIM AGRO - JACINTO MACHADO 03.451.1170005-03/01-11-2025 30-11-2025 RECEBIDOS/42251048402634000105550010000070761041242112-nfe.xml')

WindowsPath('E:/OLIM AGRO - JACINTO MACHADO 03.451.1170005-03/01-11-2025 30-11-2025 RECEBIDOS/42251048402634000105550010000071291110958490-nfe.xml')

0                      
1                      
2                      
3                      
4                      
              ...      
31196    03451117000503
31197    03451117000503
31198    03451117000503
31199    03451117000503
31200    03451117000503
Name: CNPJ Destinatário, Length: 31201, dtype: str

: 

In [ ]:
cnpj = '83286500'
nome = 'IBS e CBS'
nome = 'Fornecedores'
entradas_forn = processados.loc[processados['CNPJ Destinatário'].str.startswith(cnpj)]

fornecedores_lista = entradas_forn[['CNPJ Emitente','Razão Social Emitente', 'IE Emitente','UF Emitente','Mun Emitente','Regime Trib']]
fornecedores_lista = fornecedores_lista.rename(columns={'CNPJ Emitente': 'CNPJ',
                                                        'Razão Social Emitente': 'Razão Social',
                                                        'IE Emitente': 'Inscrição Estadual',
                                                        'UF Emitente': 'Estado',
                                                        'Mun Emitente': 'Município',
                                                        'Regime Trib': 'Regime Tributário',
                                                        })

fornecedores_lista = fornecedores_lista.drop_duplicates(subset=['CNPJ'])
display(fornecedores_lista)

In [54]:
arquivo = r'C:\Users\muril\Desktop\Vcode\Python\Projeto XML\NFSe\01-03-2026 a 31-03-2026 Recebidas\12004012284322114000148000000000002126030221997776.xml'


caminho = Path(r'C:\Users\muril\Desktop\Vcode\Python\Projeto XML\NFSe\saida')
xmls = list(caminho.rglob("*.xml"))
notas = []
ctes = []
erros = []
nfse = []

for arquivo in xmls:

    with open(arquivo, "r", encoding="utf-8") as file:
        xml = xmltodict.parse(file.read())
        inf_nfse = pegar(xml, ['NFSe', 'infNFSe'])
        emissor = pegar(inf_nfse, ['emit'])
        dps = pegar(inf_nfse, ['DPS', 'infDPS'])
        prestador = pegar(dps, ['prest'])
        tomador = pegar(dps, ['toma'])
        servico = pegar(dps, ['serv'])
        valores = pegar(inf_nfse, ['valores'])
        trib_fed = pegar(dps, ['valores','trib', 'tribFed'])
        pis_cofins = pegar(trib_fed, ['piscofins']),
        cclas = pegar( dps, ['IBSCBS', 'C'])
    

        nfse.append({
            # ---------------- IDE ----------------
            'Numero NF': pegar(inf_nfse, ['nNFSe']),
            'Série': pegar(dps, ['serie']),
            'Numero DFSe': pegar(inf_nfse, ['nDFSe']),
            'Emissão': pegar(inf_nfse, ['dhProc']),
            'Competencia': pegar(dps, ['dCompet']),
            #'Competencia': pegar(inf_nfse, ['Competencia']),
            #'Codigo Verificação': pegar(inf_nfse, ['CodigoVerificacao']),

            # ---------------- PRESTADOR ----------------
            'CNPJ Emitente': pegar(emissor, ['CNPJ']) or pegar(emissor, ['CPF']),
            #'Inscrição Municipal Emitente': pegar(prestador, ['InscricaoMunicipal']),
            'Razão Social Emitente': pegar(emissor, ['xNome']),
            'CNPJ Prestador': pegar(emissor, ['CNPJ']) or pegar(prestador, ['CPF']),

            'Município Prestador': pegar(emissor, ['enderNac', 'cMun']),
            'UF Prestador': pegar(emissor, ['enderNac', 'UF']),
            'Optante do simples': pegar(prestador, ['regTrib', 'opSimpNac']),
            'Regime Especial': pegar(prestador, ['regTrib', 'regEspTrib']),

            #'Regime Especial Tributação': pegar(inf_nfse, ['RegimeEspecialTributacao']),
            #'Optante Simples Nacional': pegar(inf_nfse, ['OptanteSimplesNacional']),
            #'Incentivador Cultural': pegar(inf_nfse, ['IncentivadorCultural']),

            # ---------------- TOMADOR ----------------
            'CNPJ Tomador': pegar(tomador, ['CNPJ']) or pegar(tomador, ['Cpf']),
            #'Inscrição Municipal Tomador': pegar(tomador, ['InscricaoMunicipal']),
            'Razão Social Tomador': pegar(tomador, ['xNome']),

            'Município Tomador': pegar(tomador, ['end', 'endNac', 'cMun']),

            # ---------------- SERVIÇO ----------------
            'Local Prestação': pegar(servico, ['locPrest', 'cLocPrestacao']),
            'Cod. Local Incidencia': pegar(inf_nfse, ['cLocIncid']),
            'Local Incidencia': pegar(inf_nfse, ['xLocIncid']),
            'Código Serviço': pegar(servico, ['cServ', 'cTribNac']),
            'Descrição Nacional': pegar(inf_nfse, ['xTribNac']),
            'Codigo NBS': pegar(servico, ['cServ','cNBS']),
            'Descrição NBS': pegar(inf_nfse, ['xNBS']),
            'Cod.Trib. Municipal': pegar(servico, ['cServ', 'cTribMun']),
            'Descrição Municipal': pegar(inf_nfse, ['xTribMun']),
            'Descrição do emissor': pegar(servico, ['cServ', 'xDescServ']),

            # ---------------- VALORES ----------------
            'Valor Serviço': pegar(dps, ['valores','vServPrest', 'vServ' ], 0),
            'Descontos': pegar(valores, ['vDescCondIncond']),

            #-----------------ISS----------------------
            'Base ISS': pegar(valores, ['vBC'], 0),
            'Aliq. ISS': pegar(valores, ['pAliqAplic'], 0),
            'Valor ISSQN': pegar(valores, ['vISSQN']),

            #---------------PIS e COFINS----------------

            'CST PIS/COFINS': pegar(trib_fed, ['piscofins', 'CST']),
            'Base PIS/COFINS': pegar(trib_fed, ['piscofins', 'vBCPisCofins']),
            'Aliq.PIS': pegar(trib_fed, ['piscofins', 'pAliqPis']),
            'Valor PIS': pegar(trib_fed, ['piscofins', 'vPis']),
            'Aliq.COFINS': pegar(trib_fed, ['piscofins', 'pAliqCofins']),
            'Valor COFINS': pegar(trib_fed, ['piscofins', 'vCofins']),
            'Tipo de Retenção PC': pegar(trib_fed, ['piscofins', 'tpRetPisCofins']),

            'IRRF': pegar(trib_fed, ['vRetIRRF']),
            'CSLL': pegar(trib_fed, ['vRetCSLL']),

            'Total retenções': pegar(valores, ['vTotalRet']),
            'Valor Liquido': pegar(valores, ['vLiq']),



            # UF
            'Finalidade IBS e CBS': pegar(dps, ['IBSCBS', 'finNFSe']),
            'Indicador de Operação': pegar(dps, ['IBSCBS', 'cIndOp']),
            'cClasTrib': pegar(dps, ['IBSCBS', 'valores', 'trib', 'gIBSCBS' ,'cClassTrib'],'Não Destacado'),

            'BC IBS e CBS': pegar(ibscbs, ['valores','vBC']),
            'Aliquota IBS UF': pegar(ibscbs, ['valores', 'uf', 'pIBSUF']),
            'Valor IBS UF': pegar(ibscbs, ['totCIBS', 'gIBS','gIBSUFTot', 'vIBSUF']),

            'Aliquota IBS UF': pegar(ibscbs, ['valores', 'mun', 'pIBSMun']),
            'Valor IBS UF': pegar(ibscbs, ['totCIBS', 'gIBS','gIBSMunTot', 'vIBSMun']),

            'Aliquota IBS UF': pegar(ibscbs, ['valores', 'fed', 'pCBS']),
            'Valor IBS UF': pegar(ibscbs, ['totCIBS', 'gCBS','vCBS']),


        })
    servicos = pd.DataFrame(nfse)
    servicos['Emissão'] = pd.to_datetime(servicos['Emissão'], utc=True).dt.strftime('%d/%m/%Y')
    servicos['Competencia'] = pd.to_datetime(servicos['Competencia'], utc=True).dt.strftime('%d/%m/%Y')
    cnpj = '83472803000176'
    break

df_saidas_nfs = servicos.loc[servicos['CNPJ Emitente'].str.startswith(cnpj)]

display(servicos['Indicador de Operação'])

0    050101
Name: Indicador de Operação, dtype: str